# Кинетическая модель мембраны с двумя разными гранями — без постулата детального равновесия

**Цель.** Посчитать стационарное отношение $P_B/P_A$ для мембраны, у которой сторона B закрыта (поток $J=0$), только из кинетических уравнений с явными константами скоростей. Условие $\mu_H^A=\mu_H^B$ **не** закладывается, второе начало тоже.

**Схема (для каждой грани $X\in\{A,B\}$):**

$$\mathrm{H_2(gas)} \;\underset{k^X_{d}}{\overset{s^X Z P_X}{\rightleftarrows}}\; 2\,\mathrm{H(surf)} \;\underset{k^X_{bs}}{\overset{k^X_{sb}}{\rightleftarrows}}\; \mathrm{H(bulk)}_X \;\overset{D/l}{\longleftrightarrow}\; \mathrm{H(bulk)}_{\bar X}$$

Скорости (на единицу площади), $\Theta$ — покрытие, $c$ — концентрация в объёме под гранью, $N_s$ — плотность мест:

* адсорбция/десорбция: $r_1^X = 2\left[s^X Z P_X (1-\Theta_X)^2 - k_d^X \Theta_X^2\right]$ (атомов H);
* поверхность → объём: $r_2^X = k_{sb}^X\,\Theta_X - k_{bs}^X\,c_X(1-\Theta_X)$;
* диффузия: $J = \dfrac{D}{l}(c_A - c_B)$.

Константы у граней A и B **произвольные и независимые**.

In [ ]:
preparser(False)
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(1)

def rates(y, p):
    # y = [P_B, th_A, c_A, c_B, th_B];  P_A фиксировано (большой резервуар)
    PB, thA, cA, cB, thB = y
    r1A = 2*(p['sA']*p['Z']*p['PA']*(1-thA)**2 - p['kdA']*thA**2)
    r2A = p['ksbA']*thA - p['kbsA']*cA*(1-thA)
    J   = p['Dl']*(cA - cB)
    r2B = p['ksbB']*thB - p['kbsB']*cB*(1-thB)
    r1B = 2*(p['sB']*p['Z']*PB*(1-thB)**2 - p['kdB']*thB**2)
    return dict(r1A=r1A, r2A=r2A, J=J, r2B=r2B, r1B=r1B)

def rhs(y, p):
    r = rates(y, p)
    return np.array([
        -r['r1B']/(2*p['VB']),                 # газ в замкнутом объёме B
        (r['r1A'] - r['r2A'])/p['Ns'],
        (r['r2A'] - r['J'])/p['h'],
        (r['J'] + r['r2B'])/p['h'],
        (r['r1B'] - r['r2B'])/p['Ns']])

def integrate(y, p, t_end, n=4000):
    # неявный Эйлер с ньютоновскими итерациями и логарифмическим шагом (система жёсткая)
    ts = np.concatenate([[0], np.logspace(-6, np.log10(t_end), n)])
    Y = [y.copy()]
    for k in range(1, len(ts)):
        dt = ts[k]-ts[k-1]; z = Y[-1].copy()
        for _ in range(30):
            F = z - Y[-1] - dt*rhs(z, p)
            Jm = np.empty((5,5)); e = 1e-9
            for i in range(5):
                dz = np.zeros(5); dz[i] = e*max(1, abs(z[i]))
                Jm[:,i] = ((z+dz) - Y[-1] - dt*rhs(z+dz, p) - F)/dz[i]
            step = np.linalg.solve(Jm, -F); z = z + step
            if np.max(np.abs(step)/np.maximum(1e-12, np.abs(z))) < 1e-12: break
        Y.append(z)
    return ts, np.array(Y)
print('модель определена')

## Рисунок: схема модели

(a) Мембрана и все стадии с их скоростями. Синие стрелки — прямые процессы, красные — обратные. (b) Энергетический профиль атома H вдоль цепочки; уровни **иллюстративные**, это не данные. Отношение давлений при $J=0$ — это произведение констант равновесия звеньев по всему пути газ A → газ B. Если уровень H(объём) не зависит от того, с какой грани мы пришли, получаем $W=1$. В профиле на рисунке этот уровень при входе через A и через B *нарочно* нарисован одинаковым, а отличаются только поверхностные ямы и барьеры: так и выглядит ситуация с $W=1$ при разных гранях.

In [ ]:
# Рисунок: схема мембраны и энергетический профиль водорода по цепочке газ A → грань A → объём → грань B → газ B
import numpy as np, matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9.5), gridspec_kw=dict(height_ratios=[1, 1.15]))

# ---------- (a) схема ----------
ax1.set_xlim(0, 13); ax1.set_ylim(0, 5); ax1.axis('off')
ax1.add_patch(Rectangle((0.2, 0.6), 2.8, 3.8, fc='#dbeafe', ec='none'))
ax1.add_patch(Rectangle((10.0, 0.6), 2.8, 3.8, fc='#dbeafe', ec='none'))
ax1.add_patch(Rectangle((3.6, 0.6), 5.8, 3.8, fc='#e5e7eb', ec='k'))
ax1.add_patch(Rectangle((3.6, 0.6), 0.35, 3.8, fc='#fca5a5', ec='k'))
ax1.add_patch(Rectangle((9.05, 0.6), 0.35, 3.8, fc='#86efac', ec='k'))
ax1.text(1.6, 4.65, 'газ A, давление $P_A$\n(резервуар)', ha='center', fontsize=11)
ax1.text(11.4, 4.65, 'газ B, давление $P_B$\n(закрытый объём или нагрузка)', ha='center', fontsize=11)
ax1.text(3.78, 0.25, 'грань A\n$\\Theta_A$', ha='center', fontsize=10, color='#b91c1c')
ax1.text(9.22, 0.25, 'грань B\n$\\Theta_B$', ha='center', fontsize=10, color='#15803d')
ax1.text(6.5, 4.0, 'объём металла', ha='center', fontsize=11)
ax1.text(5.0, 1.0, '$c_A$', fontsize=12); ax1.text(7.8, 1.0, '$c_B$', fontsize=12)
ax1.plot([4.3, 8.7], [1.8, 1.4], 'k--', lw=1)
ax1.text(6.5, 1.9, '$J = \\frac{D}{l}(c_A - c_B)$', ha='center', fontsize=11)
def arr(x1, x2, y, lab, col):
    ax1.add_patch(FancyArrowPatch((x1, y), (x2, y), arrowstyle='-|>', mutation_scale=16, color=col, lw=1.6))
    ax1.text((x1 + x2)/2, y + 0.12, lab, ha='center', fontsize=10, color=col)
arr(2.4, 3.55, 3.2, '$s^A Z P_A(1-\\Theta)^2$', 'C0'); arr(3.55, 2.4, 2.6, '$k_d^A\\Theta^2$', 'C3')
arr(4.0, 5.2, 3.2, '$k_{sb}^A\\Theta$', 'C0');      arr(5.2, 4.0, 2.6, '$k_{bs}^A c(1-\\Theta)$', 'C3')
arr(9.0, 7.8, 3.2, '$k_{sb}^B\\Theta$', 'C0');      arr(7.8, 9.0, 2.6, '$k_{bs}^B c(1-\\Theta)$', 'C3')
arr(10.6, 9.45, 3.2, '$s^B Z P_B(1-\\Theta)^2$', 'C0'); arr(9.45, 10.6, 2.6, '$k_d^B\\Theta^2$', 'C3')
ax1.set_title('(a) Последовательная цепочка стадий; константы граней A и B независимы', fontsize=12)

# ---------- (b) энергетический профиль (на один атом H) ----------
x  = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8])
lab = ['½H$_2$ (газ A)', 'барьер\nдиссоц.', 'H(ад) A', 'барьер\nподпов.', 'H(объём)', 'барьер\nподпов.', 'H(ад) B', 'барьер\nдиссоц.', '½H$_2$ (газ B)']
EA = [0.0, 0.05, -0.50, -0.05, -0.10]          # иллюстративные уровни, эВ
EB = [-0.10, 0.10, -0.38, 0.12, 0.0]
E  = EA + EB[1:]
xs = np.linspace(0, 8, 800); Es = np.interp(xs, x, E)
ax2.plot(xs, Es, color='k', lw=1.5)
for xi, Ei, li in zip(x, E, lab):
    ax2.plot(xi, Ei, 'o', color='k'); ax2.text(xi, Ei + (0.05 if Ei >= -0.2 else -0.11), li, ha='center', fontsize=9)
ax2.axhline(0, color='C0', lw=.8, ls=':')
ax2.annotate('', xy=(4, -0.10), xytext=(4, 0.0), arrowprops=dict(arrowstyle='<->', color='C3'))
ax2.text(4.15, -0.05, '$-\\Delta H_S$ (растворение)', color='C3', fontsize=10)
ax2.annotate('', xy=(8.35, 0.0), xytext=(8.35, 0.0), arrowprops=dict(arrowstyle='-'))
ax2.text(0.3, -0.58, '$K_1^A=\\frac{s^AZ}{k_d^A}$', fontsize=11, color='#b91c1c')
ax2.text(2.6, -0.58, '$K_2^A=\\frac{k_{sb}^A}{k_{bs}^A}$', fontsize=11, color='#b91c1c')
ax2.text(4.9, -0.58, '$K_2^B=\\frac{k_{sb}^B}{k_{bs}^B}$', fontsize=11, color='#15803d')
ax2.text(7.0, -0.58, '$K_1^B=\\frac{s^BZ}{k_d^B}$', fontsize=11, color='#15803d')
ax2.text(4, 0.33, 'Обход цикла газ A → объём → газ B:   $W=(P_B/P_A)_{J=0}=\\frac{K_1^A (K_2^A)^2}{K_1^B (K_2^B)^2}=(S_A/S_B)^2$', ha='center', fontsize=12,
         bbox=dict(fc='#fef9c3', ec='#ca8a04'))
ax2.text(4, 0.22, 'Детальное равновесие: уровень H(объём) один и тот же при входе с любой грани ⇒ $W=1$.   Кинетика сама по себе этого не требует.',
         ha='center', fontsize=10)
ax2.set_xlim(-0.5, 8.5); ax2.set_ylim(-0.68, 0.42); ax2.set_xticks([]); ax2.set_ylabel('энергия на атом H, эВ (иллюстративно)')
ax2.set_title('(b) Энергетический профиль. Изотермы Christmann дают только звенья $K_1$; звенья $K_2$ нужно мерить отдельно', fontsize=12)
plt.tight_layout(); plt.savefig('membrane_two_faces_scheme.png', dpi=120); plt.show()


## 1. Теорема о закрытой цепочке

В стационаре $\dot y = 0$. Сторона B закрыта, значит $r_1^B=0$. Тогда из $\dot\Theta_B=0$ следует $r_2^B=0$, из $\dot c_B=0$ — $J=0$, из $\dot c_A=0$ — $r_2^A=0$, из $\dot\Theta_A=0$ — $r_1^A=0$.

Это **только сохранение числа атомов** в последовательной цепочке без ветвлений. Никакой термодинамики здесь нет. Каждое звено в стационаре имеет нулевую *нетто*-скорость. При этом $\Theta$ и $c$ на двух гранях могут быть любыми, совместимыми с уравнениями ниже.

Из нулевых нетто-скоростей:

$$\frac{\Theta_X}{1-\Theta_X} = \sqrt{K_1^X P_X},\quad K_1^X=\frac{s^X Z}{k_d^X};\qquad c_X = K_2^X\frac{\Theta_X}{1-\Theta_X},\quad K_2^X = \frac{k^X_{sb}}{k^X_{bs}};\qquad c_A=c_B .$$

Отсюда $c_X = S_X\sqrt{P_X}$, где **эффективная константа Сивертса грани** $S_X = K_2^X\sqrt{K_1^X}$, и

$$\boxed{\;\frac{P_B}{P_A} = \left(\frac{S_A}{S_B}\right)^2 = \frac{K_1^A\,(K_2^A)^2}{K_1^B\,(K_2^B)^2}\;}$$

Проверка ниже: решаем нестационарную систему со **случайными** константами и сравниваем предел $P_B(t\to\infty)$ с формулой. Заодно проверяем, что все нетто-скорости обнуляются.

In [ ]:
def random_params():
    lg = lambda a, b: 10**rng.uniform(a, b)
    return dict(Z=1.0, PA=lg(-1, 1), sA=lg(-2, 0), sB=lg(-2, 0),
                kdA=lg(-1, 1), kdB=lg(-1, 1), ksbA=lg(-1, 1), kbsA=lg(-1, 1),
                ksbB=lg(-1, 1), kbsB=lg(-1, 1), Dl=lg(-1, 1), Ns=1.0, h=1.0, VB=1.0)

def P_B_theory(p):
    S = lambda s, kd, ksb, kbs: (ksb/kbs)*np.sqrt(s*p['Z']/kd)
    SA = S(p['sA'], p['kdA'], p['ksbA'], p['kbsA']); SB = S(p['sB'], p['kdB'], p['ksbB'], p['kbsB'])
    return p['PA']*(SA/SB)**2

worst = 0; worst_r = 0
for trial in range(8):
    p = random_params()
    y0 = np.array([p['PA'], 0.0, 0.0, 0.0, 0.0])
    ts, Y = integrate(y0, p, 1e9, n=900)
    PBnum = Y[-1, 0]; PBth = P_B_theory(p)
    r = rates(Y[-1], p); rmax = max(abs(v) for v in r.values())
    worst = max(worst, abs(PBnum/PBth - 1)); worst_r = max(worst_r, rmax)
    print(f'{trial}: P_B/P_A числ = {PBnum/p["PA"]:.5g}, формула = {PBth/p["PA"]:.5g}')
print(f'max отн. ошибка P_B = {worst:.2e}; max |нетто-скорость| в конце = {worst_r:.2e} ->',
      'OK' if worst < 1e-6 and worst_r < 1e-8 else 'FAIL')

**Итог п. 1.** Кинетика без всяких постулатов даёт $P_B \neq P_A$, как только $S_A\neq S_B$. Это не вечный двигатель: $J=0$, поток не течёт, а давление на закрытой стороне просто устанавливается другим.

Весь вопрос сводится к одному числу — **отношению Вегшайдера для цикла**

$$W \equiv \frac{S_A^2}{S_B^2} = \frac{K_1^A (K_2^A)^2}{K_1^B (K_2^B)^2}.$$

Цикл: газ → грань A → объём → грань B → газ. Детальное равновесие (через то, что концентрация H в объёме — свойство металла, а не грани) требует $W=1$. Кинетика сама по себе $W$ не фиксирует.

## 2. Может ли такая мембрана дать работу?

Пусть $W\neq1$. Откроем сторону B через нагрузку (дроссель) в резервуар с $P_A$, получится замкнутый газовый контур при одной температуре. Если $W>1$, поток $J>0$ будет идти стационарно, и газ на дросселе можно заставить совершать работу $\approx J\,kT\ln(P_B/P_A)$.

Проверка ниже: открытая сторона B, $P_B=P_A$, стационарный $J$ по той же модели. Ожидание: $J\neq0$ ровно тогда, когда $W\neq1$, а знак $J$ совпадает со знаком $\ln W$.

In [ ]:
def steady_J_open(p):
    # P_B = P_A зафиксировано; ищем стационар по th_A, c_A, c_B, th_B методом Ньютона
    q = dict(p); 
    def F(z):
        y = np.array([q['PA'], *z]); return rhs(y, q)[1:]
    z = np.array([0.5, 1.0, 1.0, 0.5])
    for _ in range(200):
        f = F(z); Jm = np.empty((4,4))
        for i in range(4):
            dz = np.zeros(4); dz[i] = 1e-8*max(1, abs(z[i])); Jm[:,i] = (F(z+dz)-f)/dz[i]
        step = np.linalg.solve(Jm, -f); lam = 1.0
        while np.any((z+lam*step)[[0,3]] <= 0) or np.any((z+lam*step)[[0,3]] >= 1) or np.any((z+lam*step)[[1,2]] < 0): lam /= 2
        z = z + lam*step
        if np.max(np.abs(f)) < 1e-14: break
    return rates(np.array([q['PA'], *z]), q)['J']

ok = True
for trial in range(8):
    p = random_params()
    W = (P_B_theory(p)/p['PA'])
    J = steady_J_open(p)
    agree = (np.sign(J) == np.sign(np.log(W))) or abs(np.log(W)) < 1e-9
    ok &= agree
    print(f'{trial}: ln W = {np.log(W):+.3f}, J(P_B=P_A) = {J:+.4e}', '' if agree else '<- знак не совпал')
# симметричный случай W = 1: разные константы, но S_A = S_B
p = random_params(); 
p['ksbB'] = p['kbsB']*(p['ksbA']/p['kbsA'])*np.sqrt((p['sA']/p['kdA'])/(p['sB']/p['kdB']))
J1 = steady_J_open(p)
print(f'W = 1 при разных гранях: J = {J1:.2e} ->', 'OK' if abs(J1) < 1e-10 else 'FAIL')
print('знак J = знак ln W:', 'OK' if ok else 'FAIL')

## 3. Что о $W$ говорят данные Christmann

Изотермы Christmann дают **только $K_1$**: равновесие газ ⇄ поверхность на каждой грани отдельно. Это половина цикла. Константы поверхность ⇄ объём $K_2^X$ из них получить нельзя.

Поскольку реальные изотермы не ленгмюровские (есть латеральное взаимодействие), используем **локальную** $K_1$ при $\Theta=0.5$: тогда $\Theta/(1-\Theta)=1$ и $K_1^X = 1/P_X(\Theta=0.5)$.

Тогда $W=1$ эквивалентно требованию к подповерхностным константам:

$$\frac{K_2^{110}}{K_2^{111}} = \sqrt{\frac{K_1^{111}}{K_1^{110}}} = \sqrt{\frac{P_{110}(0.5)}{P_{111}(0.5)}} .$$

Это и есть **проверяемое предсказание постулата**. Если измеренное отношение $K_2$ (энергии подповерхностных мест, TDS-пики β/подповерхностного водорода на Pd(110) и Pd(111)) отличается от этого числа больше погрешности, $W\neq1$.

In [ ]:
TORR = 133.322368
def read_bln(f):
    rows = []
    for line in open(f):
        if '"' in line or not line.strip(): continue
        rows.append([float(x) for x in line.replace(' ', ',').split(',')[:3]])
    return np.array(rows)
DIR = 'didger'
def P_half(face, dphi):
    d = read_bln(f'{DIR}/Pd_{face}_p.bln'); out = {}
    for t in sorted(set(d[:,2])):
        m = d[:,2] == t; th = d[m,1]/dphi; lnP = np.log(d[m,0]*TORR); o = np.argsort(th)
        if th.min() <= 0.5 <= th.max(): out[t+273.15] = np.exp(np.interp(0.5, th[o], lnP[o]))
    return out
P110, P111 = P_half('110', 360.0), P_half('111', 180.0)
R = 8.314462618
print(' T, K   P110(0.5), Па  P111(0.5), Па   K1_111/K1_110   требуемое K2_110/K2_111')
Ts = sorted(set(P110) & set(P111))
for T in Ts:
    r = P110[T]/P111[T]
    print(f'{T:6.1f}   {P110[T]:.3e}     {P111[T]:.3e}     {r:8.3f}          {np.sqrt(r):.3f}')
# в энергетическом виде: ΔE_2 = E_2(110) - E_2(111), требуемая для W=1
lr = [0.5*np.log(P110[T]/P111[T]) for T in Ts]
b, a = np.polyfit(1/np.array(Ts), lr, 1)
print(f'\nln(K2_110/K2_111)_треб = {a:+.2f} {b:+.0f}/T  ->  ΔH_2(110−111) ≈ {-b*R/1000:+.1f} кДж/моль '
      f'(энтальпийная часть требования W=1)')

## 4. Выводы

**А. Получено без постулатов (проверено в ячейках 1–2)**

* В закрытой цепочке газ ⇄ поверхность ⇄ объём ⇄ поверхность ⇄ газ стационарное отношение $P_B/P_A = W$. Здесь $W$ — произведение констант равновесия звеньев по циклу. Это следует только из сохранения числа атомов.
* При открытой стороне и $P_A=P_B$ стационарный поток $J\neq0$ тогда и только тогда, когда $W\neq1$, а знак $J$ равен знаку $\ln W$. Значит, **мембрана с $W\neq1$ — это именно вечный двигатель 2-го рода** в смысле изотермического циклического контура. Кинетическая модель такое допускает: её уравнения согласованы при любом $W$.

**Б. Что добавляет детальное равновесие**

* Только одно утверждение: $W=1$ для любой пары граней. Термодинамически это значит, что $S_X$ — свойство объёма металла, а не грани. В кинетику это не встроено, это внешнее ограничение на константы.

**В. Как решить вопрос эмпирически**

* Изотермы Christmann фиксируют $K_1$ каждой грани (п. 3). Постулат $W=1$ при этом даёт **численное предсказание** для отношения подповерхностных констант $K_2^{110}/K_2^{111}$ и для соответствующей разности энтальпий.
* Проверка требует независимых измерений $K_2$ (подповерхностного/абсорбционного равновесия) на тех же гранях. Второй вариант — прямо измерить зависимость константы Сивертса от ориентации грани, через которую насыщался образец: $W = S_{110}^2/S_{111}^2$.
* Пока таких данных нет, ни «$W=1$», ни «$W\neq1$» из имеющихся файлов не следует.

## 5. Что говорит литература из папок `hydrogen_metal_iteraction/` и `ref/`

**1. Livshitz 1979 (Vacuum 29, 103), §7.2 и последний абзац §7.1.** Прямо утверждается: «для газа с распределением состояний, соответствующим температуре мембраны, проницаемость мембраны в обоих направлениях одинакова как при идентичных, так и при неидентичных границах». Это и есть $W=1$, но в статье оно **постулировано**, а не выведено: см. сноску к §4, где «равновесная система» определена как изотермичная, со всеми процессами тепловыми. Степень сжатия $\beta=v_0\varkappa_0/(v_l\varkappa_l)$ (41) отлична от 1 только при неравновесном газе с одной стороны (42).

**2. Livshits, Notkin, Samartsev 1990 (J. Nucl. Mater. 170, 79), §2.1–2.4.** Константа рекомбинации записывается как $k_r = Z\alpha/S^2$, где $S$ — константа Сивертса **металла**. Это детальное равновесие для каждой поверхности отдельно, с одним и тем же объёмным $S$, то есть ровно условие $W=1$ из п. 1 этого ноутбука. В §2.4 сказано, что асимметрия проницаемости «присуща неравновесным условиям, тогда как проницаемость для молекулярного водорода одинакова в обоих направлениях». Там же: стационарная концентрация в «горячем» газе зависит от поверхности, **в отличие от** равновесной $c=S\sqrt p$.

**3. Hatano, Watanabe, Livshits et al. 2007 (J. Chem. Phys. 127, 204707).** Это **экспериментальная проверка** условия $W=1$ для отдельной поверхности. Коэффициент прилипания $\alpha_{H_2}$ Nb, покрытого кислородом, измерялся двумя независимыми способами:
* напрямую — по стационарному потоку проникновения, $T_g=300$ K, менялась только $T_S$;
* через константу десорбции $k_r$ (из времени запаздывания) по детальному равновесию $\alpha = Z^{-1}S^2k_r$ с **объёмной** растворимостью $S$ из литературы.

Авторы сообщают о согласии; аналогичное согласие отмечено для других металлов, покрытых монослоем неметалла (их ссылки 1, 22, 26, 27). На языке этого ноутбука это значит, что для данной поверхности $K_1K_2^2 = S_{\rm bulk}^2$, то есть «$S$, измеренная через поверхность», совпадает с объёмной. Если это так для обеих сторон, $W=1$. Там же есть оговорка: из V(111), покрытого O и S, десорбируют гипертермальные D$_2$ с энергией в 3–4 раза больше $2kT_S$. С детальным равновесием это совместимо: активированное прилипание даёт и «горячую» десорбцию.

**4. Livshits et al. 2002, 2004; Cherkez 2020.** Используется полезная асимметрия $\alpha_{down}\gg\alpha_{up}$, например шероховатость на выходе или ионное распыление на входе. Фактор асимметрии $A=\alpha_{up}/(\alpha_{up}+\alpha_{down})$ (Livshits 2004, ур. 1) прямо выведен из детального равновесия. Асимметрия здесь работает **только для надтеплового** входного потока ($\xi\sim1$); тепловой газ с обеих сторон в этих работах насос не даёт.

**5. Christmann 1988, Bonzel 1987.** Детальное равновесие обсуждается на уровне *распределений* (по колебательным состояниям, по углам). Christmann приводит пример, где перенос детального равновесия на колебательные распределения десорбирующих молекул **не** предсказывает правильно вероятность адсорбции по колебательным состояниям. Это нарушение не детального равновесия, а предположения, что десорбирующий поток термализован. Для интегральных констант $K_1,K_2$ вывод не меняется.

**Итог по литературе.** Во всех найденных работах $W=1$ для теплового газа принимается: детальное равновесие используется как инструмент. Единственная прямая экспериментальная проверка этого условия для отдельной поверхности (Hatano 2007 и ссылки там) даёт согласие в пределах точности порядка десятков процентов. Измерения $W$ на одной мембране с двумя разными поверхностями при тепловом H$_2$ с обеих сторон я в этих папках **не нашёл**. Это и есть недостающий эксперимент.

## 6. Полное чтение литературы (62 PDF из `hydrogen_metal_iteraction/` и `ref/`)

Подробные отчёты по пакетам, с таблицами параметров и дословными цитатами со страницами, лежат в папке `lit_review/`: `A_livshits.md`, `B_permeation.md`, `C_christmann1988.md`, `D_surface.md`, `E_books_dft.md`. Монография Бокштейна не читалась.

### Главное
**Прямого измерения $W$ нет ни в одной работе.** Никто не измерял константу Сивертса через грань A и через грань B одной мембраны. Нет и опыта с тепловым H$_2$ равного $p,T$ по обе стороны асимметричной мембраны. Все количественные модели закладывают $W=1$ по построению (одна объёмная $S$, $k_r=Z\alpha/S^2$).

### Что ближе всего к проверке
| Работа | Что сделано | Что это проверяет |
|---|---|---|
| Hatano 2007 (Nb–O) | $\alpha$ из стационарного потока совпала с $Z^{-1}S^2k_r$ при табличной $S$; проницаемость трубки измерена в двух направлениях, разница в пределах погрешности | детальное равновесие на **одной** поверхности; грани одинаковые, $W$ не проверяется |
| Musyaev 2005 (Nb), 2008 (V) | асимметрия $\alpha_{out}/\alpha_{in}\sim10$–$10^3$; при выключенном атомизаторе «давления возвращаются к исходным» | качественно: асимметрия $\alpha$ не даёт самонакачки; равных давлений специально не создавали, порог не указан |
| Livshits 1997, 1998 | с холодным атомизатором проникновения нет; сжатие $10^3$–$10^4$ только с плазмой или горячими атомами | то же, качественно |
| Nakamura 2000 | асимметрия $k_r\approx2$ по ТДС, **при допущении** равных концентраций у граней | в модель заложено $W=1$ |
| Alimov 2014 | $S$(V) с Pd-покрытием ≈ литературной $S$(V) на других поверхностях | слабый довод, что покрытие не меняет равновесную $S$ |

### Явные утверждения $W=1$ как постулата
* Livshits, Notkin, Samartsev 1990, с. 82: «any kinetic formalism … should meet one requirement stipulated by the principles of thermodynamics: no regular gas flow may exist under isothermic conditions, if hydrogen pressures are equal on the both sides of a membrane». Там же критика работ Swansiger & Bastasz 1979, патента Behr et al. и Ионова [34]: из их рассуждений следовал бы поток при равных давлениях. **Эти работы стоит найти: это прямые предшественники вашей постановки.**
* Livshitz 1979, конец §7.1; Livshits 2002 FST; 2023: $p_{d,max}$ не зависит от выходной поверхности, это прямое следствие $W=1$.
* Christmann 1988: детальное равновесие используется на каждой грани ($\nu_2$ из равенства скоростей адсорбции и десорбции); поверхность меняет скорость, но не равновесие (H$_2$:HD:D$_2$ = 1:2:1 на любой активной грани).

### Что могло бы говорить против (и почему не говорит)
* Kubiak et al. (у Christmann, с. 89): H$_2$, проникший через Cu(110) и Cu(111), десорбирует с избытком $v=1$ примерно в 100 раз против Больцмана. Здесь не выполнено детальное равновесие, перенесённое на *распределения* десорбирующего потока. Потока при равных давлениях не измеряли. По памяти (не проверено): позднее Rettner, Michelsen, Auerbach (~1992–95) показали согласие адсорбции и десорбции с детальным равновесием по отдельным состояниям для H$_2$/Cu(111).
* Comsa (Pd(100)): горячие D$_2$ рекомбинируют из объёма, минуя хемосорбционную яму. Это другой путь, но для $W$ важны константы равновесия, а не путь.
* Livshits 1990, с. 82, сноска: разница проницаемостей «up to about 60%» в переходном режиме. Это при $p\ne0$ с одной стороны и вакууме с другой, не при равных давлениях.
* Опыты неизотермичны: газ комнатной температуры, мембрана 500–1000 °C. Сами авторы (1990, с. 83) признают, что асимметрия проницаемости для H$_2$ здесь законна и должна исчезать при $T_g=T_s$.

### Чего не хватает для $W$ на Pd(110)/Pd(111)
* $K_1$ по граням известны: $E_{ad}$ = 24.4 ккал/моль для Pd(110), 20.8 для Pd(111), 24.5 для Pd(100) (Conrad 1974; Behm 1980), что согласуется с нашим $q_{st}$ из изотерм.
* $K_2$ не измерен ни для одной грани. Behm 1983 прямо пишет, что абсолютные уровни подповерхностного состояния Pd(110) неизвестны; объёмные уровни на рис. 54 у Christmann взяты из объёмных данных. Поверхность меняет только барьер входа, то есть $W=1$ заложено в саму диаграмму.

### Вывод
Литература единодушно принимает $W=1$. Экспериментально она показывает, что весь наблюдавшийся направленный транспорт требует неравновесного входного потока. Количественного опыта, который ограничивал бы $|W-1|$ при тепловом газе и одной температуре, в этих 62 работах нет. Нужны следующие источники:
1. Swansiger & Bastasz 1979 и патент Behr et al. — ссылки [37], [38] у Livshits 1990.
2. Ионов [34] у Livshits 1990.
3. Kompaniets & Kurdyumov, *Prog. Surf. Sci.* 17 (1984) 75.
4. Amandusson et al. 2000 и 2001 — Pd с модифицированной одной стороной.
5. Rettner, Michelsen, Auerbach — детальное равновесие для отдельных состояний H$_2$/Cu.

## 7. Kubiak et al. (у Christmann 1988, с. 89): проверка «доказательства отклонения от детального равновесия»

**Цитата сверена с PDF обзора (с. 89) дословно.** Ссылка [274]: G.D. Kubiak, G.O. Sitz, R.N. Zare, *J. Chem. Phys.* **83** (1985) 2538.

Опыт:
* атомарный водород проникал через кристаллы Cu(110) и Cu(111), десорбция в вакуум при $T_s$ = 850 и 960 K;
* вращательная энергия молекул — 80–90 % от $kT_s$;
* отношение заселённостей $v{=}1$ : $v{=}0$ примерно в 100 раз больше больцмановского при $T_s$.

По аннотации к Kubiak et al., *J. Chem. Phys.* **81**, 6397 (1984): превышение примерно в 50 раз для Cu(110) и примерно в 100 раз для Cu(111). Полные тексты 1984 и 1985 гг. не прочитаны (закрыты подпиской).

**Чем закончилось.** Колебательный «перегрев» десорбирующих молекул по микрообратимости требует, чтобы колебательно возбуждённые H$_2$ прилипали лучше. Это потом подтвердили прямые пучковые опыты (Rettner, Michelsen, Auerbach и др., 1990-е; ссылки по Gross & Scheffler, arXiv:mtrl-th/9607006). Для H$_2$/Cu(111) получены пороги адсорбции: $E_0\approx0.6$ эВ ($v{=}0$), 0.3 эВ ($v{=}1$), 0.1 эВ ($v{=}2$) (*J. Chem. Phys.* 102, 4625, 1995). Десорбционные данные, обработанные через детальное равновесие, воспроизводят тренды адсорбционных измерений (*Surf. Sci.*, 1993). Всё это — по аннотациям, полные тексты не прочитаны.

**Почему это не довод против $W=1$:**
1. Вывод Kubiak 1985 сделан до измерения адсорбции по отдельным колебательным состояниям. Когда её измерили, детальное равновесие для H$_2$/Cu подтвердилось.
2. Даже в исходной форме речь идёт о *распределении* потока десорбции по состояниям, а не о равновесных константах. Неравновесное распределение десорбирующего потока **требуется** детальным равновесием, если адсорбция активирована по колебательной координате.
3. Опыт односторонний: десорбция в вакуум, без обратного потока и без равных давлений. Для $W$ нужны интегральные $K_1$, $K_2$.

**Что нужно для дословной сверки:**
* Kubiak, Sitz, Zare, JCP 83, 2538 (1985), doi:10.1063/1.449300;
* Kubiak, Sitz, Zare, JCP 81, 6397 (1984);
* Michelsen, Auerbach, JCP 94, 7502 (1991), «A critical examination of data on the dissociative adsorption and associative desorption of hydrogen at copper surfaces».


## 8. Проверка «необратимой мембраны» Merrill–Crowley (US 3 344 582, 1967) расчётом

Вопрос: совместимо ли сильное выпрямление (поток в одну сторону при перепаде давлений много больше, чем в другую) с $W=1$, то есть с детальным равновесием?

**1. Линейная модель Лифшица (второй порядок, без насыщения поверхности).** Перебор $\alpha_0/\alpha_L$ и $D/L$ даёт выпрямление не больше **≈1.6**, что согласуется с оценкой «до 60 %» (Livshits 1990, Самарцев 1985). Патент так не объясняется.

**2. Полная цепочка с насыщением мест** (модель этого ноутбука, $(1-\Theta)$ в адсорбции и в обмене с объёмом). Ниже 400 случайных мембран, в каждой условие $W=1$ наложено точно. Результат: выпрямление до **~1400**, при этом $J(P,P)=0$ с машинной точностью.

Механизм на найденном примере. На грани B глубокая хемосорбционная яма ($K_1^B$ велико, $k_d^B$ мало), и при потоке наружу через B поверхность насыщается, $\Theta_B\approx0.999$:
* **A→B** (газ со стороны A, вакуум со стороны B): атомы выходят через B, а рекомбинационная десорбция с B медленная, поэтому поток мал;
* **B→A**: газ легко садится на B, а выход через A быстрый, поэтому поток большой.

Это обычный кинетический «диод» на нелинейности насыщения, как p–n-переход: асимметрия есть при $\Delta p\neq0$ и обращается в ноль при $\Delta p=0$.

**Вывод.** Сильное выпрямление патента Merrill–Crowley **не противоречит** детальному равновесию. Для него достаточно насыщения поверхности на одной стороне, и $W\neq1$ не нужно. Поэтому «необратимая мембрана» не свидетельствует против второго начала: такой диод не качает газ при равных давлениях.

In [ ]:

def PB_of_J(J,p,PA):
    a=2*(p['sA']*PA-p['kdA']); b=-4*p['sA']*PA; c=2*p['sA']*PA-J
    rts=np.roots([a,b,c]) if abs(a)>1e-300 else np.array([-c/b])
    t=[r.real for r in rts if abs(r.imag)<1e-12 and 0<=r.real<=1]
    if not t: return np.nan
    tA=t[0]
    cA=(p['ksbA']*tA-J)/(p['kbsA']*(1-tA))
    if cA<0: return np.nan
    cB=cA-J/p['g']
    if cB<0: return np.nan
    tB=(p['kbsB']*cB-J)/(p['ksbB']+p['kbsB']*cB)
    if not 0<=tB<1: return np.nan
    return (-J/2+p['kdB']*tB**2)/(p['sB']*(1-tB)**2)

def J_of(p,PA,PB):
    f=lambda J:PB_of_J(J,p,PA)
    Js=np.concatenate([-np.logspace(6,-12,600),[0],np.logspace(-12,6,600)])
    v=np.array([f(J) for J in Js]); ok=np.isfinite(v); Js,v=Js[ok],v[ok]
    k=np.where(np.diff(np.sign(v-PB))!=0)[0]
    if len(k)==0: return np.nan
    a,b=Js[k[0]],Js[k[0]+1]
    for _ in range(100):
        m=(a+b)/2
        if np.sign(f(m)-PB)==np.sign(f(a)-PB): a=m
        else: b=m
    return (a+b)/2

rng=np.random.default_rng(0); best=0; bp=None; n=0
for t in range(400):
    lg=lambda: 10**rng.uniform(-4,4)
    p=dict(sA=lg(),kdA=lg(),ksbA=lg(),kbsA=lg(),sB=lg(),kdB=lg(),kbsB=lg(),g=lg())
    K1A=p['sA']/p['kdA']; K2A=p['ksbA']/p['kbsA']; K1B=p['sB']/p['kdB']
    p['ksbB']=p['kbsB']*np.sqrt(K1A*K2A**2/K1B)          # принудительно W = 1
    P=10**rng.uniform(-3,3)
    f=J_of(p,P,0.0); r=J_of(p,0.0,P)                    # A→вакуум и B→вакуум
    if np.isfinite(f) and np.isfinite(r) and f*r<0:
        n+=1; R=max(abs(f/r),abs(r/f))
        if R>best: best,bp=R,(p,P,f,r)
p,P,f,r=bp
W=p['sA']/p['kdA']*(p['ksbA']/p['kbsA'])**2/(p['sB']/p['kdB']*(p['ksbB']/p['kbsB'])**2)
print(f'{n} мембран; макс. выпрямление при W=1: {best:.0f}  (W={W:.12f})')
print(f'J(A→B)={f:.3e}, J(B→A)={r:.3e}, J(P,P)={J_of(p,P,P):.1e}')
print('W=1, выпрямление ≫ 2, нет потока при равных p:', 'OK' if best>100 and abs(J_of(p,P,P))<1e-10 else 'FAIL')


## 9. Карта выпрямления при $W=1$

Грань A «каталитическая», все её константы равны 1. На грани B задаются два параметра:
* сила связи $K_1^B=s^B/k_d^B$;
* общий кинетический множитель $\beta$: $s^B=\beta\sqrt{K_1^B}$, $k_d^B=\beta/\sqrt{K_1^B}$.

Константа $k_{sb}^B$ подобрана так, чтобы $W=1$ выполнялось точно. $R=J(A\to B)/|J(B\to A)|$, на противоположной стороне вакуум. Расчётная сетка 13×13, белые клетки — точки, где решатель не нашёл физического корня.

**Результаты:**
* $|J(P,P)|\le1.4\cdot10^{-16}$ во всех проверенных точках;
* $R$ лежит в пределах $5.6\cdot10^{-4}$…$76$, то есть выпрямление до ~1800 раз;
* **медленная и сильно связывающая грань B** (низ справа, $\beta\ll1$, $K_1^B\gg1$) пропускает газ в основном **в сторону A**. Выход через B ограничен рекомбинацией с насыщенной поверхности;
* **быстрая и сильно связывающая B** (верх справа) даёт обратное направление;
* выпрямление растёт с давлением и насыщается (правая панель) — это признак нелинейности заполнения мест; при $P\to0$ $R\to1$, как в линейной теории.

**Сравнение с патентом Merrill–Crowley.** В патенте газ проходит со стороны катализатора (Pd) на некаталитическую (Au, сталь), а обратно почти не идёт. В модели это соответствует грани B с **малым** $\beta$ и **слабой** связью ($K_1^B\lesssim1$), левая нижняя область. Там $R$ = 0.26…0.83, то есть выпрямление не больше ~4, причём преимущественное направление B→A, обратное патенту. **Сильного** выпрямления в направлении патента в этой области модель не даёт.

Значит, для количественного объяснения патента нужен механизм сверх модели: например, градиент температуры (пример 7 патента), кинетика с барьером, зависящим от покрытия, или просто очень малый обратный поток, который ниже чувствительности манометра. Без чисел из патента выбрать между ними нельзя. Детального равновесия ни один из этих вариантов не нарушает.

In [ ]:
# Карта выпрямления (использует PB_of_J, J_of из раздела 8)
def mk(K1B, beta):
    p = dict(sA=1.0, kdA=1.0, ksbA=1.0, kbsA=1.0, kbsB=1.0, g=1.0)
    p['sB'] = beta*np.sqrt(K1B); p['kdB'] = beta/np.sqrt(K1B)
    p['ksbB'] = p['kbsB']*np.sqrt(1.0/K1B)           # W = 1
    return p
def R(K1B, beta, P):
    p = mk(K1B, beta); f = J_of(p, P, 0.0); r = J_of(p, 0.0, P)
    return f/(-r) if np.isfinite(f) and np.isfinite(r) and r != 0 else np.nan
K = np.logspace(-2, 6, 13); Bt = np.logspace(-4, 2, 13); Ps = np.logspace(-4, 4, 13)
M  = np.array([[R(k, b, 1.0) for k in K] for b in Bt])
Mp = np.array([R(1e4, 1e-2, pp) for pp in Ps])
Jeq = max(abs(J_of(mk(k, b), 1, 1)) for k in (1e-2, 1e2, 1e6) for b in (1e-4, 1, 1e2))
print(f'max|J(P,P)| = {Jeq:.1e} ->', 'OK' if Jeq < 1e-12 else 'FAIL')
print(f'R: {np.nanmin(M):.2e} … {np.nanmax(M):.2e}')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
im = ax[0].pcolormesh(np.log10(K), np.log10(Bt), np.log10(M), cmap='RdBu_r', vmin=-3.5, vmax=3.5, shading='nearest')
ax[0].contour(np.log10(K), np.log10(Bt), np.log10(M), levels=[0], colors='k')
plt.colorbar(im, ax=ax[0], label='log10 R')
ax[0].set(xlabel='log10 K1_B', ylabel='log10 β', title='Выпрямление при W=1, P=1')
ax[1].semilogx(Ps, np.log10(Mp), 'o-'); ax[1].axhline(0, color='k', lw=.5)
ax[1].set(xlabel='P', ylabel='log10 R', title='K1_B=1e4, β=1e-2')
for a in ax: a.grid(ls=':')
plt.tight_layout(); plt.savefig('rectification_map.png', dpi=110); plt.show()